# Session 02 - Data Collection and Problem Framing

**Learning outcome:** frame an AI problem as a measurable data task with clear assumptions and risks.

This notebook uses a small workplace-style dataset of service requests. Your goal is not to train a model yet. Your goal is to inspect the data, define a target variable, identify leakage and governance risks, and choose success metrics.

## 1. Imports and data loading

We use **Pandas** for tabular inspection and **NumPy** for simple calculations. Keep this notebook in GitHub so changes are visible and reproducible.

In [26]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path('../data/service_requests_session02.csv')
df = pd.read_csv(DATA_PATH)
df.head()

,case_id,submitted_date,channel,service_area,customer_type,postcode_area,priority_flag,days_open,updates_count,previous_cases,assigned_team,late_status_2024,satisfaction_score,resolved_on_time
0,SR-1001,2024-01-08,Web,Housing,Resident,BS1,0,2,1,0,Team A,0,4,1
1,SR-1002,2024-01-09,Phone,Benefits,Resident,BS5,1,8,4,2,Team B,1,2,0
2,SR-1003,2024-01-10,Email,Planning,Business,BS8,0,5,2,1,Team C,0,3,1
3,SR-1004,2024-01-11,Web,Waste,Resident,BS3,0,1,1,0,Team A,0,5,1
4,SR-1005,2024-01-12,Phone,Housing,Resident,BS4,1,12,5,3,Team B,1,1,0


## 2. First inspection

Ask: What does one row represent? Which columns are inputs, and which columns could be outcomes?

In [27]:
print('Rows, columns:', df.shape)
print('Columns:')
print(df.columns.tolist())
print('Data types:')
print(df.dtypes)
print('Missing values:')
print(df.isna().sum())

Rows, columns: (40, 14)
Columns:
['case_id', 'submitted_date', 'channel', 'service_area', 'customer_type', 'postcode_area', 'priority_flag', 'days_open', 'updates_count', 'previous_cases', 'assigned_team', 'late_status_2024', 'satisfaction_score', 'resolved_on_time']
Data types:
case_id                 str
submitted_date          str
channel                 str
service_area            str
customer_type           str
postcode_area           str
priority_flag         int64
days_open             int64
updates_count         int64
previous_cases        int64
assigned_team           str
late_status_2024      int64
satisfaction_score    int64
resolved_on_time      int64
dtype: object
Missing values:
case_id               0
submitted_date        0
channel               0
service_area          0
customer_type         0
postcode_area         0
priority_flag         0
days_open             0
updates_count         0
previous_cases        0
assigned_team         0
late_status_2024      0
satisfacti

### Mini-task 1: Define the prediction moment

Write one sentence: **At what moment would a user need the prediction?**

Example: "When a service request is submitted, estimate whether it is likely to be resolved on time so the team can prioritise support.

In [28]:
prediction_moment = "When a service request is submitted, predict whether it will be resolved on time."
prediction_moment

'When a service request is submitted, predict whether it will be resolved on time.'

## 3. Define the target variable

A target is the measurable outcome we want the model to learn. Here, a reasonable candidate is `resolved_on_time`.

A useful target should be:
- measurable from historical data,
- meaningful for the organisation,
- available only after the outcome happens,
- ethically acceptable to predict and act upon.

In [29]:
target = 'resolved_on_time'

print(df[target].value_counts())
print('\nTarget rate:')
print(df[target].mean())

resolved_on_time
1    24
0    16
Name: count, dtype: int64

Target rate:
0.6


### Mini-task 2: Alternative targets

Suggest two alternative targets from this dataset. For each, note whether it is a classification or regression problem.

In [30]:
alternative_targets = pd.DataFrame({
    'candidate_target': ['days_open', 'satisfaction_score'],
    'problem_type': ['Regression', 'Regression or ordinal classification'],
    'business_question': ['How long will this case take?', 'What satisfaction score might this case receive?']
})
alternative_targets

,candidate_target,problem_type,business_question
0,days_open,Regression,How long will this case take?
1,satisfaction_score,Regression or ordinal classification,What satisfaction score might this case receive?


## 4. Identify features and leakage risks

Leakage happens when the training data contains information that would not be available at the prediction moment. For example, `days_open`, `late_status_2024`, and `satisfaction_score` may reveal or follow from the outcome.

In [31]:
candidate_features = [c for c in df.columns if c not in ['case_id', target]]

availability = pd.DataFrame({
    'column': candidate_features,
    'available_at_submission?': [
        'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'maybe', 'yes', 'yes', 'no', 'no'
    ],
    'reason': [
        'Submitted date is known at the start.',
        'Channel is known at the start.',
        'Service area is known at the start.',
        'Customer type is known at the start.',
        'Broad postcode area may be known, but check privacy policy.',
        'Priority flag is known at triage.',
        'Days open is only known later.',
        'Updates count may change during the case.',
        'Previous cases may be known from CRM history.',
        'Assigned team may be known after routing; depends on prediction moment.',
        'Late status is effectively the answer.',
        'Satisfaction score is collected after resolution.'
    ]
})
availability

,column,available_at_submission?,reason
0,submitted_date,yes,Submitted date is known at the start.
1,channel,yes,Channel is known at the start.
2,service_area,yes,Service area is known at the start.
3,customer_type,yes,Customer type is known at the start.
4,postcode_area,yes,"Broad postcode area may be known, but check pr..."
5,priority_flag,yes,Priority flag is known at triage.
6,days_open,no,Days open is only known later.
7,updates_count,maybe,Updates count may change during the case.
8,previous_cases,yes,Previous cases may be known from CRM history.
9,assigned_team,yes,Assigned team may be known after routing; depe...


### Mini-task 3: Choose safe features

Create a list named `safe_features`. Include only columns available at the prediction moment. Be ready to justify each choice.

In [32]:
safe_features = [
    'submitted_date', 'channel', 'service_area', 'customer_type',
    'postcode_area', 'priority_flag', 'previous_cases'
]

leaky_or_uncertain = sorted(set(candidate_features) - set(safe_features))
print('Safe features:', safe_features)
print('Excluded:', leaky_or_uncertain)

Safe features: ['submitted_date', 'channel', 'service_area', 'customer_type', 'postcode_area', 'priority_flag', 'previous_cases']
Excluded: ['assigned_team', 'days_open', 'late_status_2024', 'satisfaction_score', 'updates_count']


## 5. Basic data quality checks

Good problem framing includes evidence about whether the data is usable: completeness, duplicates, type consistency, class balance, and suspicious columns.

In [33]:
quality_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_rate': df.isna().mean(),
    'unique_values': df.nunique()
}).sort_values('missing_rate', ascending=False)
quality_summary

,missing_count,missing_rate,unique_values
case_id,0,0.0,40
submitted_date,0,0.0,40
channel,0,0.0,3
service_area,0,0.0,5
customer_type,0,0.0,2
postcode_area,0,0.0,9
priority_flag,0,0.0,2
days_open,0,0.0,14
updates_count,0,0.0,5
previous_cases,0,0.0,5


In [34]:
duplicates = df.duplicated(subset=['case_id']).sum()
print('Duplicate case IDs:', duplicates)

# Simple profile of categorical fields
for col in ['channel', 'service_area', 'customer_type', 'postcode_area']:
    print(f'{col}')
    print(df[col].value_counts())

Duplicate case IDs: 0
channel
channel
Web      19
Phone    11
Email    10
Name: count, dtype: int64
service_area
service_area
Housing      9
Waste        9
Benefits     8
Planning     7
Licensing    7
Name: count, dtype: int64
customer_type
customer_type
Resident    26
Business    14
Name: count, dtype: int64
postcode_area
postcode_area
BS1    5
BS5    5
BS3    5
BS2    5
BS8    4
BS4    4
BS6    4
BS7    4
BS9    4
Name: count, dtype: int64


## 6. Train / validation / test thinking, without modelling yet

Even before training, decide how you would evaluate fairly. A common split is train for fitting, validation for choosing features and thresholds, and test for one final check. Time-based problems often need time-based splits so that the model is tested on later cases, not randomly mixed future and past cases.

In [35]:
df_sorted = df.sort_values('submitted_date').reset_index(drop=True)
n = len(df_sorted)
train_end = int(n * 0.60)
val_end = int(n * 0.80)

split = np.array(['train'] * n)
split[train_end:val_end] = 'validation'
split[val_end:] = 'test'

df_sorted['split'] = split
pd.crosstab(df_sorted['split'], df_sorted[target], normalize='index').round(2)

resolved_on_time,0,1
split,,
test,0.38,0.62
train,0.42,0.58
valid,0.38,0.62


## 7. Success metrics and business risk

For this task, a model could support prioritisation. A useful metric must match the consequence of mistakes:

- **False positive:** predicted at risk, but it would have resolved on time. This may waste staff attention.
- **False negative:** predicted safe, but it later misses the target. This may harm service levels or customer experience.

Before training, agree which mistake is more costly.

In [36]:
metric_plan = pd.DataFrame({
    'stakeholder_goal': [
        'Find cases likely to miss target',
        'Avoid overwhelming teams with false alarms',
        'Maintain service fairness across groups'
    ],
    'candidate_metric': ['Recall for at-risk cases', 'Precision for at-risk cases', 'Slice checks by service_area/channel'],
    'why_it_matters': [
        'Missing an at-risk case may cause service failure.',
        'Too many false alarms makes the system ignored.',
        'Overall performance can hide uneven outcomes.'
    ]
})
metric_plan

,stakeholder_goal,candidate_metric,why_it_matters
0,Find cases likely to miss target,Recall for at-risk cases,Missing an at-risk case may cause service fail...
1,Avoid overwhelming teams with false alarms,Precision for at-risk cases,Too many false alarms makes the system ignored.
2,Maintain service fairness across groups,Slice checks by service_area/channel,Overall performance can hide uneven outcomes.


## 8. Governance and ethics checklist

Complete this before modelling:

1. What lawful basis or policy basis permits use of the data?
2. Is personal data minimised? Could broad postcode area be removed or aggregated?
3. Could the target encode historic bias or operational under-resourcing?
4. Who is accountable for decisions assisted by the model?
5. How will users challenge incorrect predictions?

In [37]:
risk_register = pd.DataFrame({
    'risk': [
        'Leakage from columns only known after resolution',
        'Unfair prioritisation by postcode area',
        'Target reflects historic service delays',
        'Users over-trust model score'
    ],
    'likelihood': ['High', 'Medium', 'Medium', 'Medium'],
    'impact': ['High', 'High', 'Medium', 'High'],
    'mitigation': [
        'Remove days_open, late_status_2024, satisfaction_score before training.',
        'Test performance by postcode area and consider excluding or aggregating.',
        'Review target with operations and frontline staff.',
        'Use decision support language and human review.'
    ]
})
risk_register

,risk,likelihood,impact,mitigation
0,Leakage from columns only known after resolution,High,High,"Remove days_open, late_status_2024, satisfacti..."
1,Unfair prioritisation by postcode area,Medium,High,Test performance by postcode area and consider...
2,Target reflects historic service delays,Medium,Medium,Review target with operations and frontline st...
3,Users over-trust model score,Medium,High,Use decision support language and human review.


## 9. Mini-project deliverable

Create a short problem-framing memo with:

- problem statement,
- target definition,
- safe feature list,
- leakage exclusions,
- proposed train/validation/test split,
- success metrics,
- top three risks and mitigations.

Save it as Markdown in your GitHub repository.

In [38]:
memo = f'''# Problem Framing Memo - Service Request Triage

## Problem statement
{prediction_moment}

## Target
`{target}`: 1 if the case was resolved on time, 0 otherwise.

## Safe features
{', '.join(safe_features)}

## Leakage exclusions
{', '.join(leaky_or_uncertain)}

## Split strategy
Use a time-based split: oldest 60% train, next 20% validation, newest 20% test.

## Success metrics
Prioritise recall for at-risk cases, then precision to control false alarms. Check slices by service area and channel.

## Risks
See risk register above.
'''
print(memo)

# Problem Framing Memo - Service Request Triage

## Problem statement
When a service request is submitted, predict whether it will be resolved on time.

## Target
`resolved_on_time`: 1 if the case was resolved on time, 0 otherwise.

## Safe features
submitted_date, channel, service_area, customer_type, postcode_area, priority_flag, previous_cases

## Leakage exclusions
assigned_team, days_open, late_status_2024, satisfaction_score, updates_count

## Split strategy
Use a time-based split: oldest 60% train, next 20% validation, newest 20% test.

## Success metrics
Prioritise recall for at-risk cases, then precision to control false alarms. Check slices by service area and channel.

## Risks
See risk register above.

